# dcc_10

データ収集とクリーニング

10-データ操作：時刻データの扱い、集計、結合処理

---

## 【要点まとめ】 データの操作：時刻データの扱い、集計、結合処理

### **時刻データの扱い、集計、結合処理**

#### **時刻データの扱い**
ログデータや売上データには必ず時刻情報が含まれます。Pandasは強力な時系列データ処理機能を持っています。

``pd.to_datetime()``: ただの文字列（例: '2023-11-01'）を、Pandasが計算可能な「datetime型」に変換します。

In [ ]:
import pandas as pd
# サンプルデータを作成（日本語の形式を含む日付文字列）
df_time = pd.DataFrame({'日付文字列': ['2023年11月01日', '2023年11月02日']})
df_time

In [ ]:
# formatを指定してdatetime型に変換
# %Yは4桁の年, %mは2桁の月, %dは2桁の日 を表す
df_time['日付'] = pd.to_datetime(df_time['日付文字列'], format='%Y年%m月%d日')
df_time

``.dt`` アクセサ: datetime型に変換した後、``.dt`` を使うことで、その日付から「年」「月」「日」「曜日」などの情報を簡単に抽出できます。

In [ ]:
# .dtアクセサで「年」と「曜日名」を抽出
df_time['年'] = df_time['日付'].dt.year
df_time['曜日'] = df_time['日付'].dt.day_name()
df_time

#### **集計 (Group-By)**

データを特定のカテゴリごとに要約することは、分析の基本です。

``groupby()``: データを特定のキー（例: 「カテゴリ」列）でグループに分割します。<br>
集計関数 (``.sum()``, ``.mean()``など): 分割したグループごとに、指定した列の合計や平均を計算します。

In [ ]:
# サンプルデータを作成
data = {'カテゴリ': ['果物', '野菜', '果物', '野菜'],
        '売上': [100, 150, 200, 50]}
df = pd.DataFrame(data)
df

In [ ]:
# 'カテゴリ' ごとに '売上' を合計し、新しいDataFrame（Series）として df_agg に格納
df_agg = df.groupby('カテゴリ')['売上'].sum()
df_agg

#### **結合 (Merge, Concat)**

複数のデータソースを統合するために、DataFrame同士を結合します。

``pd.merge()`` (横方向の結合): 2つのDataFrameを、共通のキー（例: 「顧客ID」）をもとに横方向に結合します。
- ``how='inner'`` (デフォルト): 両方のDataFrameに存在するキーのデータのみを残します（**内部結合**）。
- ``how='outer'`` : どちらかのDataFrameに存在するキーのデータはすべて残します（**外部結合**）。片方にしか存在しないキーの場合、もう一方のデータは ``NaN``（欠損値）になります。

In [ ]:
# サンプルデータ（2つのDataFrame）を作成
df_sales = pd.DataFrame({'顧客ID': ['A01', 'A02', 'A03'],
                         '売上': [1000, 2000, 3000]})
df_customer = pd.DataFrame({'顧客ID': ['A01', 'A02', 'A04'],
                            '名前': ['佐藤', '鈴木', '高橋']})

df_sales

In [ ]:
df_customer

In [ ]:
# 内部結合 (inner merge)
# 出力: (両方に存在する A01, A02 のみ)
pd.merge(df_sales, df_customer, on='顧客ID', how='inner')

In [ ]:
# 外部結合 (outer merge)
# 出力: (A01, A02, A03, A04 のすべてが残る)
pd.merge(df_sales, df_customer, on='顧客ID', how='outer')

``pd.concat()`` (縦方向の結合): 同じ列構造を持つ複数のDataFrameを、単純に縦に連結します。月ごとの売上ファイル（4月.csv, 5月.csv, ...）を1つのファイルにまとめる際などに使います。
- 縦方向 (``axis=0``): (デフォルト) 同じ列構造を持つDataFrameを縦に連結します。月ごとの売上ファイル（4月.csv, 5月.csv, ...）を1つにまとめる際に使います。
- 横方向 (``axis=1``): **インデックス（行ラベル）** に基づいて横に連結します。キー列の値は考慮しません。

In [ ]:
# サンプルデータ（2つのDataFrame）を作成
df_april = pd.DataFrame({'月': [4, 4], '売上': [100, 150]})
df_may = pd.DataFrame({'月': [5, 5], '売上': [200, 180]})

In [ ]:
df_april

In [ ]:
df_may

In [ ]:
# 縦に連結 (axis=0)
# ignore_index=True を指定すると、元のインデックスを無視して新しい連番(0, 1, 2, 3...)を振り直す
pd.concat([df_april, df_may], ignore_index=True)

In [ ]:
# サンプルデータ（横連結用）
df_info1 = pd.DataFrame({'年齢': [25, 30]}, index=['A01', 'A02'])
df_info2 = pd.DataFrame({'性別': ['男', '女']}, index=['A01', 'A02'])

# 横に連結 (axis=1) - インデックス 'A01', 'A02' に基づいて結合
pd.concat([df_info1, df_info2], axis=1)

In [ ]:
df_info1

In [ ]:
df_info2

### **まとめ**

**実践的応用**:
- **時刻データ**: ``.dt``アクセサは、ログ解析（例: アクセス集中時間帯の特定）や時系列分析（例: 曜日別の売上傾向分析）の基盤技術です。
- ``groupby``: データ分析の心臓部です。「商品カテゴリごとの平均売上」や「顧客年代別の購入頻度」など、ビジネス上の意思決定に必要な集計表は、ほぼすべてこの``groupby``操作から生まれます。
- ``merge``: データベースの世界における「**JOIN**」操作と全く同じ概念です。「売上テーブル」と「顧客マスタテーブル」を結合し、分析用のリッチなデータセットを作る、データ分析の基本中の基本です。
- ``merge``の最適化: ``merge``は重い処理になりがちですが、結合キーを ``category``型に変換しておいたり、事前に重複キーを削除したり、必要な列だけを選択して結合したりすることで、処理速度を改善できる場合があります。


**ポイント**:
- **SQLとの関連**: ``pd.merge()``の``how``引数（``inner, outer``）は、データベース言語``SQL``の``INNER JOIN, OUTER JOIN``といった概念と深く関連しています。同様に、Pandasの``groupby()``は``SQLのGROUP BY``句に相当します。
- ``merge`` vs ``concat``: この2つの関数の違いを正確に理解することが重要です。
  - ``merge``: 「**キー列に基づく結合（SQLライク）**」です。内部的には結合キーをハッシュ化して照合（ハッシュジョイン）するような複雑な処理を行います。キー列の値に基づいてデータを紐付けたい場合に必須です。
  - ``concat``: 「**単純な連結**」です。DataFrameのブロックを単純に貼り合わせる（インデックスや列ラベルを合わせる）簡易な処理です。列構造が同じデータを縦に連結する場合、``concat``が``merge``よりも圧倒的に高速です。

## 10-課題
- dcc_10: 時刻データ処理と結合処理・集計

### dcc_10
``pandas``の ``read_excel`` 関数を使い、``data_10``内のエクセルファイル``20251102-1000_RoomA.xlsx``の``ASR`` シートをDataFrameに読み込んでください。
- ``end[sec]`` 列と ``begin[sec]`` 列の差を計算し、新しい ``duration[sec]``（発言時間） 列を作成して、平均発言時間を計算しましょう。
- また、このエクセルファイルは、録音日-開始時刻_ルーム名という命名規則によってファイル名が決められています。はじめに、このファイル名から開始時刻を読み取り、``begin[sec]`` 列の秒データを加算することで、実際の発話開始時刻 ``Timestamp`` 列を作成してください。ここから、最初の発言開始時刻と最後の発言終了時刻を求めましょう。
- また、学生番号と座席の対応表（``student_mapping.xlsx``）を、 ``df_student_map`` として読み込み、``pd.merge()`` を使って ``actor``（座席） 列をキーに結合して、 ``StudentID`` ごとにグループ化し、``duration[sec]`` の合計（合計発言時間）を計算することで、それぞれの学生の発言時間を計算しましょう。

これらの結果をまとめた、データ分析レポートを完成させてください。

In [ ]:
20251102-1000_RoomA

In [ ]:
# --- ライブラリのインポート ---
# pandasライブラリをインポート
# ヒント: pd という別名(alias)を付けるのが一般的です。
import pandas as pd

In [ ]:
# --- データの読み込み ---
filepath = 'data_10/20251102-1000_RoomA.xlsx'

# 'ASR'シートのデータを読み込みます
df = pd.read_excel(filepath, sheet_name='ASR') # 解答

In [ ]:
df.head()

In [ ]:
# 'end[sec]' 列と 'begin[sec]' 列の差を計算し、新しい 'duration[sec]' 列を作成します。
# ヒント: DataFrameの列同士はそのまま四則演算が可能です。
df['duration[sec]'] = df['end[sec]'] - df['begin[sec]']#修正してください。
df.head()

In [ ]:
# osモジュールをインポートして、ファイルパスからファイル名を取得
import os
filename = os.path.basename(filepath)
start_time_str = filename.split('_')[0] #修正してください。
start_time_str 

In [ ]:
# pd.to_datetime() と format='%Y%m%d-%H%M' を使います
recording_start_time = pd.to_datetime(start_time_str, format='%Y%m%d-%H%M')
print(f"録音開始時刻: {recording_start_time.strftime('%Y年%m月%d日 %H時%M分%S.%f秒')}")

In [ ]:
# --- 発話開始時刻の計算 ---
# 'begin[sec]' 列（秒）を開始時刻に加算して、実際の発話開始時刻 'Timestamp' 列を作成します。
# ヒント: pd.to_timedelta(..., unit='s') を使います。
df['Timestamp'] = recording_start_time + pd.to_timedelta(df['begin[sec]'], unit='s')
df.tail()

In [ ]:
# --- 録音終了時刻の計算 ---
# 'end[sec]' 列の最大値（最後の発言の終了秒）を取得します。
# 録音開始時刻に、最後の発言の終了秒を加算して、録音終了時刻を計算します。
max_end_sec = df['end[sec]'].max()
recording_end_time = recording_start_time + pd.to_timedelta(max_end_sec, unit='s')
print(f"録音終了時刻: {recording_end_time.strftime('%Y年%m月%d日 %H時%M分%S.%f秒')}")

In [ ]:
# 別のExcelファイル（学生マスタ）を読み込みます。
mapping_filepath = 'data_10/student_mapping.xlsx'

df_student_map = pd.read_excel(mapping_filepath)
# print("--- 学生マスタ ---")
df_student_map

In [ ]:
# pd.merge() を使って、元の df と df_student_map を 'actor' 列をキーに結合します。
# ヒント: how='left' を指定すると、元のdfの行（欠損値含む）をすべて残せます。
df_merged = pd.merge(df, df_student_map, on='actor', how='left') #修正してください。
df_merged.head()

In [ ]:
# 結合後の df_merged を 'StudentID' ごとにグループ化し、'duration[sec]' の合計を計算します。
# ヒント: .groupby('キー列')['集計したい列'].sum()
df_merged.groupby('StudentID')['duration[sec]'].sum()  #修正してください。

#### **データ分析レポート**

ファイル名から録音日は ____ で、録音開始時刻は ____ であった。また、最後の発言が終了した実質的な録音終了時刻は ____ と計算できた。学生の発言時間を集計すると、``s12345``の学生は ____ 秒、``s12346``の学生は ____ 秒、``s12347``の学生は ____ 秒であることがわかった。